# Benchmark sin ground-truth — detector × tracker

## ¿Para qué sirve este notebook?

Sirve para **decidir, con datos, qué configuración del pipeline es la mejor** sin
tener anotaciones manuales (ground-truth). La anotación del equipo en Roboflow sigue
pausada, así que en lugar de medir *exactitud vs. humano* (mAP, MOTA, mIoU — que
necesitan GT), medimos **métricas sin referencia**: consistencia física/temporal de
las predicciones y costo computacional.

El resultado es una **tabla comparativa** de las configuraciones, con tres ejes:

1. **Eficiencia** — FPS y VRAM pico (el "filtro de la realidad": descarta lo inviable).
2. **Trayectoria** — qué tan lógicas/estables son los tracks (longitud, fragmentación,
   suavidad).
3. **Máscara** — qué tan estable es la segmentación en el tiempo (IoU temporal, jitter).

## ¿Por qué es un benchmark honesto (sin fuga de datos)?

El detector YOLO (`best.pt`) se entrenó en *fase_1* con auto-labels SAM3 de los **103
videos NO-testing** (splits 0+1, split por video, anti-leak). Los **20 videos de
testing (`split=2`) quedaron intocados** → YOLO nunca los vio. Por eso corremos sobre
el testing: es justo para ambos detectores (ni SAM3-only ni YOLO→SAM3 tienen ventaja
de memorización).

## ¿Qué se ejecuta?

**5 configuraciones reales** sobre **5 videos del testing** elegidos al azar con
semilla fija (reproducible). Cada video se corre **completo** (duran < 2 min).

## Requisitos (correr EN EL POD, con GPU)

- Código al día (`git pull` en `develop`); si `import src.eval` falla, `pip install -e .`.
- `.env` con `CONFIG_FILENAME=01_yolo_sam3_config.json` (trae las secciones `yolo` y
  `botsort`).
- Pesos YOLO en `assets/yolo/best.pt` (los usan las configs `yolo_sam3`).
- Modelo SAM3 en `assets/sam3` y los 5 videos seeded bajo `data/raw`.

Con eso, basta **ejecutar las celdas de arriba a abajo**.

## Paso 1 — Definir las 5 configuraciones y seleccionar los videos

### Las configuraciones (qué combinaciones existen de verdad)

El pipeline tiene dos motores: `mode="segmentation"` (solo SAM3-texto, sin tracking) y
`mode="tracking"` (detector inyectable × tracker inyectable, con `obj_id` estable). Las
combinaciones **reales** son:

| Detección/segmentación | Sin tracking | ByteTrack | BoT-SORT |
|---|---|---|---|
| **SAM3 (texto)** `sam3_text` | ✅ (baseline) | ✅ | ✅ |
| **YOLO→SAM3** `yolo_sam3` | ❌ no existe | ✅ | ✅ |

- **`sam3_text`**: SAM3 segmenta directo por *prompt de texto* ("robot", "orange ball").
- **`yolo_sam3`**: YOLO da cajas rápidas → SAM3 segmenta por *box-prompt* dentro de
  ellas (YOLO localiza, SAM3 segmenta).
- **ByteTrack**: tracker ligero (Kalman + IoU). **BoT-SORT**: añade compensación de
  movimiento de cámara (GMC), más robusto pero más pesado.

**¿Por qué solo 5 y no 6?** En `mode="segmentation"` el `detector` **se ignora** (ese
modo siempre usa SAM3-texto; YOLO no está cableado ahí). Por eso "YOLO→SAM3 sin
tracking" no existe: sería idéntico al baseline SAM3-texto. Se omite para no gastar una
corrida en una fila duplicada.

### Los videos

`benchmark_videos(n=5, seed=42)` lee `db_metadata.csv`, filtra el `split==2` (testing) y
muestrea 5 videos con semilla fija → **misma lista en cada corrida** (comparaciones
reproducibles entre configs y entre días).

In [ ]:
from src.core.batch import run_batch
from src.eval.benchmark import (
    aggregate_config,
    benchmark_videos,
    comparison_table,
    write_table,
)

videos = benchmark_videos(n=5, seed=42)
print("videos del benchmark (split=2, seed=42):")
for v in videos:
    print(" ", v)

# (label, mode, detector, tracker). tracker=None solo en el baseline sin tracking.
# Nota: en segmentation el detector se ignora -> el baseline es SAM3-texto puro.
CONFIGS = [
    ("sam3_text+none",      "segmentation", "sam3_text", None),
    ("sam3_text+bytetrack", "tracking",     "sam3_text", "bytetrack"),
    ("sam3_text+botsort",   "tracking",     "sam3_text", "botsort"),
    ("yolo_sam3+bytetrack", "tracking",     "yolo_sam3", "bytetrack"),
    ("yolo_sam3+botsort",   "tracking",     "yolo_sam3", "botsort"),
]
print(f"\n{len(CONFIGS)} configuraciones a evaluar.")

## Paso 2 — Correr cada configuración y medir

### Cómo lo hace

Para cada config se llama a `run_batch`, que corre la inferencia de los 5 videos
(cargando SAM3 una sola vez por config) y devuelve un **resumen por video** que ya
incluye el **timing** instrumentado (`fps`, `peak_vram_mb`) y la ruta del JSON de
salida. Parámetros clave:

- `include_masks=True` — embebe la máscara (COCO-RLE) en el JSON, necesario para las
  métricas de máscara. Cuesta muy poco (~×1.03 en tiempo; el smoke lo midió).
- `sampling="all"` — video completo, sin acotar frames.
- `render_video=False` — no generamos mp4 (solo el dato); más rápido.
- `overwrite=True` — reprocesa aunque exista un JSON previo.

Luego `aggregate_config` **lee los JSON recién escritos** y calcula las métricas de
calidad por video, las promedia y les funde el timing → **una fila por config**.

### Por qué "una config a la vez"

Todas las configs escriben en la misma ruta `outputs/inference/<stem>/` (el JSON no se
versiona por config). Si corriéramos las 5 y luego leyéramos, solo quedaría en disco la
última. Por eso el bucle **corre una config y la agrega de inmediato** (con los JSON
frescos) antes de pasar a la siguiente, que sobrescribe.

> Las métricas de trayectoria/máscara requieren `obj_id` estable ⇒ solo salen en las 4
> configs con tracking. El baseline `sam3_text+none` reporta solo eficiencia
> (trayectoria/máscara quedan N/A). Esto es esperado, no un error.

In [ ]:
rows = []
for label, mode, detector, tracker in CONFIGS:
    print(f"\n===== {label}  ({mode}) =====")
    summary = run_batch(
        mode=mode,
        videos=videos,
        detector=detector,
        tracker=tracker,
        sampling="all",      # video completo
        include_masks=True,  # RLE en el JSON -> métricas de máscara
        render_video=False,  # solo el dato, sin mp4
        overwrite=True,      # reprocesa en cada config
    )
    # Agregar AHORA, con los JSON frescos, antes de que la siguiente config sobrescriba.
    row = aggregate_config(label, summary)
    rows.append(row)
    print("fila:", row)

## Paso 3 — Tabla comparativa

`comparison_table` arma el DataFrame (una fila por config, columnas ordenadas) y
`write_table` lo persiste como CSV en `outputs/benchmark/comparison.csv` (git-ignored).
El CSV queda como evidencia versionable del experimento.

In [ ]:
df = comparison_table(rows)
out = write_table(df)
print(f"tabla escrita en: {out}")
df

## Cómo leer la tabla

| Columna | Qué mide | Mejor cuando |
|---|---|---|
| `fps` | throughput (frames/seg) | **mayor** (más rápido) |
| `peak_vram_mb` | VRAM pico del proceso | **menor** (menos memoria) |
| `tracklet_len` | frames promedio que vive un `obj_id` | **mayor** (menos fragmentación) |
| `frag_rate` | proxy de cambios de ID (un ID muere y otro nace cerca y justo después) | **menor** |
| `smoothness` | varianza de la aceleración del centroide | **menor** (movimiento más físico) |
| `mask_iou` | IoU de la máscara del mismo objeto entre frames consecutivos | **mayor** (máscara estable) |
| `com_jitter` | temblor del centro de masa de la máscara entre frames | **menor** |

### Advertencias de interpretación (importante)

- Son métricas **sin ground-truth**: miden **consistencia y eficiencia**, NO exactitud
  vs. humano. Un `frag_rate` bajo no garantiza que el ID sea *correcto* (podría estar
  fusionando dos robots distintos). Úsalas como evidencia comparativa, no como verdad.
- `tracklet_len`/`frag_rate`/`smoothness` no interpolan frames faltantes; son un proxy
  válido **comparando configs sobre los mismos videos** (que es justo lo que hacemos).
- La **decisión final es humana**: la tabla informa el balance eficiencia↔consistencia,
  no dictamina un "ganador" automático.

### Hipótesis de ingeniería a contrastar

Se espera que **YOLO→SAM3** sea más rápido (YOLO filtra el espacio antes de SAM3) y que
**BoT-SORT** reduzca fragmentación frente a ByteTrack en cámara en movimiento, a costa
de más cómputo. El "sweet spot" típico suele ser **YOLO→SAM3 + ByteTrack**. La tabla
confirma o refuta esto con datos de nuestros videos.